In [1]:
import sys
import csv
from typing import Optional

print("Versión de Python en este entorno de Colab:")
print(sys.version)
print("\nMódulos disponibles: csv ->", csv.__name__, "| typing.Optional ->", Optional)


Versión de Python en este entorno de Colab:
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

Módulos disponibles: csv -> csv | typing.Optional -> typing.Optional


**Contexto**  
Se realizará un gestor de clientes, rutas y pedidos para Rutas del Bajío S.A. de C.V.
La empresa transporta mercancía entre León, Irapuato, Celaya, Salamanca y Guanajuato Capital.

**Análisis**  
CLIENTE → ENTIDAD  
Atributos: (id_cliente, nombre, ciudad, telefono)

RUTA → ENTIDAD  
Atributos: (id_ruta, origen, destino, distancia_km, tiempo_horas, transportista, disponible)

PEDIDO → ENTIDAD  
Atributos: (folio, id_cliente, producto, cantidad, origen, destino, fecha, estado, id_ruta)

Un cliente tiene muchos pedidos; cada pedido pertenece a un cliente.
Una ruta atiende muchos pedidos; un pedido tiene cero o una ruta.
El nombre del transportista se guarda en la ruta, como en el caso.
La disponibilidad se indica manualmente; no se calcula la capacidad de camiones.

Estados: pendiente → en_transito → entregado; pendiente → cancelado.
Cancelar es posible únicamente antes de salir. Un pedido entregado no se reabre.

**Forma de trabajo**  
Como en la plantilla: constructores, to_dict/from_dict, gestor con listas y CSV,
CREATE, READ, UPDATE y DELETE, con un ejemplo después de cada etapa.
Todo el código del sistema está en este notebook; no importa archivos del proyecto.
En Colab, sube este archivo `.ipynb` y ejecuta las celdas de arriba hacia abajo.
No requiere instalar paquetes. Los CSV se guardan en `datos_notebook/`.
Para conservarlos fuera de la sesión de Colab, descárgalos desde el panel Archivos.
Las ampliaciones de la clase por celda siguen el método de la plantilla;
se recrea el gestor después de cada ampliación para usar sus nuevos métodos.


In [2]:
# Definir las entidades y sus constructores, como en la plantilla.
"""Entidades del caso. Solo modelan y validan datos; no abren archivos."""
from datetime import date
import math
import re

CIUDADES = ("León", "Irapuato", "Celaya", "Salamanca", "Guanajuato Capital")
ESTADOS = ("pendiente", "en_transito", "entregado", "cancelado")


def texto(valor, campo):
    if not isinstance(valor, str) or not valor.strip():
        raise ValueError(f"{campo} debe contener texto.")
    return valor.strip()


def clave(valor, prefijo, digitos):
    valor = texto(valor, "Clave").upper()
    if not re.fullmatch(rf"{prefijo}-[0-9]{{{digitos},}}", valor):
        raise ValueError(f"Clave inválida: se espera {prefijo}-{'0' * digitos}.")
    return valor


def validar_ciudad(valor):
    valor = texto(valor, "Ciudad")
    for nombre in CIUDADES:
        if nombre.casefold() == valor.casefold():
            return nombre
    raise ValueError(f"Ciudad inválida. Opciones: {', '.join(CIUDADES)}.")


def entero_positivo(valor):
    if isinstance(valor, bool) or not re.fullmatch(r"[0-9]+", str(valor).strip()):
        raise ValueError("La cantidad debe ser un número entero positivo.")
    numero = int(valor)
    if numero <= 0:
        raise ValueError("La cantidad debe ser mayor que cero.")
    return numero


def decimal_positivo(valor, campo):
    if isinstance(valor, bool):
        raise ValueError(f"{campo} debe ser un número positivo.")
    try:
        numero = float(valor)
    except (ValueError, TypeError):
        raise ValueError(f"{campo} debe ser un número positivo.") from None
    if not math.isfinite(numero) or numero <= 0:
        raise ValueError(f"{campo} debe ser un número finito mayor que cero.")
    return numero


def fecha_iso(valor):
    valor = texto(valor, "Fecha")
    if not re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", valor):
        raise ValueError("La fecha debe usar el formato AAAA-MM-DD.")
    try:
        date.fromisoformat(valor)
    except ValueError:
        raise ValueError("La fecha no existe en el calendario.") from None
    return valor


class Cliente:
    """Representa un cliente; no gestiona archivos ni otros clientes."""

    def __init__(self, id_cliente: str, nombre: str, ciudad: str, telefono: str) -> None:
        self.id_cliente = clave(id_cliente, "CLI", 3)
        self.nombre = texto(nombre, "Nombre")
        self.ciudad = validar_ciudad(ciudad)
        self.telefono = texto(telefono, "Teléfono")
        if not re.fullmatch(r"[+0-9 ()-]+", self.telefono) or not 10 <= len(re.sub(r"\D", "", self.telefono)) <= 15:
            raise ValueError("El teléfono debe contener entre 10 y 15 dígitos.")


class Ruta:
    """Representa una ruta fija y su transportista responsable."""

    def __init__(self, id_ruta: str, origen: str, destino: str, distancia_km: float,
                 tiempo_horas: float, transportista: str, disponible: bool = True) -> None:
        self.id_ruta = clave(id_ruta, "RUTA", 2)
        self.origen = validar_ciudad(origen)
        self.destino = validar_ciudad(destino)
        if self.origen == self.destino:
            raise ValueError("El origen y el destino deben ser distintos.")
        self.distancia_km = decimal_positivo(distancia_km, "Distancia")
        self.tiempo_horas = decimal_positivo(tiempo_horas, "Tiempo")
        self.transportista = texto(transportista, "Transportista")
        if isinstance(disponible, str):
            if disponible.lower() not in ("true", "false"):
                raise ValueError("Disponible debe ser True o False.")
            disponible = disponible.lower() == "true"
        if not isinstance(disponible, bool):
            raise ValueError("Disponible debe ser True o False.")
        self.disponible = disponible


class Pedido:
    """Representa un pedido de traslado de mercancía."""

    def __init__(self, folio: str, id_cliente: str, producto: str, cantidad: int,
                 origen: str, destino: str, fecha: str, estado: str = "pendiente",
                 id_ruta: Optional[str] = None) -> None:
        self.folio = clave(folio, "PED", 4)
        self.id_cliente = clave(id_cliente, "CLI", 3)
        self.producto = texto(producto, "Producto")
        self.cantidad = entero_positivo(cantidad)
        self.origen = validar_ciudad(origen)
        self.destino = validar_ciudad(destino)
        if self.origen == self.destino:
            raise ValueError("El origen y el destino deben ser distintos.")
        self.fecha = fecha_iso(fecha)
        if estado not in ESTADOS:
            raise ValueError("Estado inválido.")
        self.estado = estado
        self.id_ruta = clave(id_ruta, "RUTA", 2) if id_ruta else None
        if estado in ("en_transito", "entregado") and self.id_ruta is None:
            raise ValueError("Un pedido en tránsito o entregado requiere una ruta.")


cliente_prueba = Cliente("CLI-010", "Constructora Guanajuato S.A.", "Guanajuato Capital", "473-100-2233")
ruta_prueba = Ruta("RUTA-03", "León", "Guanajuato Capital", 55, 1.5, "Julián Pérez")
pedido_prueba = Pedido("PED-0001", "CLI-010", "Láminas de acero calibre 20", 40,
                       "León", "Guanajuato Capital", "2026-09-15")
print(cliente_prueba.id_cliente, cliente_prueba.nombre, cliente_prueba.ciudad)
print(ruta_prueba.id_ruta, ruta_prueba.transportista)
print(pedido_prueba.folio, pedido_prueba.producto, pedido_prueba.estado)


CLI-010 Constructora Guanajuato S.A. Guanajuato Capital
RUTA-03 Julián Pérez
PED-0001 Láminas de acero calibre 20 pendiente


In [3]:
# Se retoman las clases para convertir sus registros a diccionario y CSV.

class Cliente:
    """Representa un cliente; no gestiona archivos ni otros clientes."""

    def __init__(self, id_cliente: str, nombre: str, ciudad: str, telefono: str) -> None:
        self.id_cliente = clave(id_cliente, "CLI", 3)
        self.nombre = texto(nombre, "Nombre")
        self.ciudad = validar_ciudad(ciudad)
        self.telefono = texto(telefono, "Teléfono")
        if not re.fullmatch(r"[+0-9 ()-]+", self.telefono) or not 10 <= len(re.sub(r"\D", "", self.telefono)) <= 15:
            raise ValueError("El teléfono debe contener entre 10 y 15 dígitos.")

    def to_dict(self) -> dict:
        """Convierte la entidad en una fila para csv.DictWriter."""
        return {
            "id_cliente": self.id_cliente,
            "nombre": self.nombre,
            "ciudad": self.ciudad,
            "telefono": self.telefono,
        }

    @classmethod
    def from_dict(cls, datos: dict):
        """Reconstruye la entidad desde una fila de csv.DictReader."""
        return cls(**datos)

    def __str__(self) -> str:
        """Devuelve una representación legible del registro."""
        return "[" + " - ".join(str(valor) for valor in self.to_dict().values()) + "]"


class Ruta:
    """Representa una ruta fija y su transportista responsable."""

    def __init__(self, id_ruta: str, origen: str, destino: str, distancia_km: float,
                 tiempo_horas: float, transportista: str, disponible: bool = True) -> None:
        self.id_ruta = clave(id_ruta, "RUTA", 2)
        self.origen = validar_ciudad(origen)
        self.destino = validar_ciudad(destino)
        if self.origen == self.destino:
            raise ValueError("El origen y el destino deben ser distintos.")
        self.distancia_km = decimal_positivo(distancia_km, "Distancia")
        self.tiempo_horas = decimal_positivo(tiempo_horas, "Tiempo")
        self.transportista = texto(transportista, "Transportista")
        if isinstance(disponible, str):
            if disponible.lower() not in ("true", "false"):
                raise ValueError("Disponible debe ser True o False.")
            disponible = disponible.lower() == "true"
        if not isinstance(disponible, bool):
            raise ValueError("Disponible debe ser True o False.")
        self.disponible = disponible

    def to_dict(self) -> dict:
        """Convierte la entidad en una fila para csv.DictWriter."""
        return {
            "id_ruta": self.id_ruta,
            "origen": self.origen,
            "destino": self.destino,
            "distancia_km": self.distancia_km,
            "tiempo_horas": self.tiempo_horas,
            "transportista": self.transportista,
            "disponible": self.disponible,
        }

    @classmethod
    def from_dict(cls, datos: dict):
        """Reconstruye la entidad desde una fila de csv.DictReader."""
        return cls(**datos)

    def __str__(self) -> str:
        """Devuelve una representación legible del registro."""
        return "[" + " - ".join(str(valor) for valor in self.to_dict().values()) + "]"


class Pedido:
    """Representa un pedido de traslado de mercancía."""

    def __init__(self, folio: str, id_cliente: str, producto: str, cantidad: int,
                 origen: str, destino: str, fecha: str, estado: str = "pendiente",
                 id_ruta: Optional[str] = None) -> None:
        self.folio = clave(folio, "PED", 4)
        self.id_cliente = clave(id_cliente, "CLI", 3)
        self.producto = texto(producto, "Producto")
        self.cantidad = entero_positivo(cantidad)
        self.origen = validar_ciudad(origen)
        self.destino = validar_ciudad(destino)
        if self.origen == self.destino:
            raise ValueError("El origen y el destino deben ser distintos.")
        self.fecha = fecha_iso(fecha)
        if estado not in ESTADOS:
            raise ValueError("Estado inválido.")
        self.estado = estado
        self.id_ruta = clave(id_ruta, "RUTA", 2) if id_ruta else None
        if estado in ("en_transito", "entregado") and self.id_ruta is None:
            raise ValueError("Un pedido en tránsito o entregado requiere una ruta.")

    def to_dict(self) -> dict:
        """Convierte la entidad en una fila para csv.DictWriter."""
        return {
            "folio": self.folio,
            "id_cliente": self.id_cliente,
            "producto": self.producto,
            "cantidad": self.cantidad,
            "origen": self.origen,
            "destino": self.destino,
            "fecha": self.fecha,
            "estado": self.estado,
            "id_ruta": self.id_ruta,
        }

    @classmethod
    def from_dict(cls, datos: dict):
        """Reconstruye la entidad desde una fila de csv.DictReader."""
        return cls(**datos)

    def __str__(self) -> str:
        """Devuelve una representación legible del registro."""
        return "[" + " - ".join(str(valor) for valor in self.to_dict().values()) + "]"


pedido_prueba = Pedido("PED-0001", "CLI-010", "Láminas de acero calibre 20", 40,
                       "León", "Guanajuato Capital", "2026-09-15")
datos = pedido_prueba.to_dict()
print("to_dict():", datos)
pedido_reconstruido = Pedido.from_dict(datos)
print("from_dict() + __str__:", pedido_reconstruido)


to_dict(): {'folio': 'PED-0001', 'id_cliente': 'CLI-010', 'producto': 'Láminas de acero calibre 20', 'cantidad': 40, 'origen': 'León', 'destino': 'Guanajuato Capital', 'fecha': '2026-09-15', 'estado': 'pendiente', 'id_ruta': None}
from_dict() + __str__: [PED-0001 - CLI-010 - Láminas de acero calibre 20 - 40 - León - Guanajuato Capital - 2026-09-15 - pendiente - None]


In [4]:
# Desarrollar el gestor: constructor, listas y persistencia en archivos CSV.

from pathlib import Path

class GestorRutasBajio:
    CAMPOS_CLIENTES = ["id_cliente", "nombre", "ciudad", "telefono"]
    CAMPOS_RUTAS = ["id_ruta", "origen", "destino", "distancia_km", "tiempo_horas", "transportista", "disponible"]
    CAMPOS_PEDIDOS = ["folio", "id_cliente", "producto", "cantidad", "origen", "destino", "fecha", "estado", "id_ruta"]

    def __init__(self, directorio: str = "datos_notebook") -> None:
        self.directorio = Path(directorio)
        self.clientes = []
        self.rutas = []
        self.pedidos = []
        self.ultimo_folio = 0
        self.cargar()

    def _leer(self, nombre, campos, clase) -> list:
        try:
            with open(self.directorio / nombre, "r", newline="", encoding="utf-8-sig") as archivo:
                lector = csv.DictReader(archivo, strict=True)
                if lector.fieldnames != campos:
                    raise ValueError(f"Encabezados incorrectos en {nombre}.")
                registros = []
                for numero, fila in enumerate(lector, 2):
                    if None in fila or any(valor is None for valor in fila.values()):
                        raise ValueError(f"Fila incompleta en {nombre}, fila {numero}.")
                    try:
                        registros.append(clase.from_dict(fila))
                    except (ValueError, TypeError) as error:
                        raise ValueError(f"{nombre}, fila {numero}: {error}") from error
                return registros
        except FileNotFoundError:
            return []

    def _escribir(self, nombre, campos, filas) -> None:
        self.directorio.mkdir(parents=True, exist_ok=True)
        ruta = self.directorio / nombre
        temporal = ruta.with_suffix(".tmp")
        try:
            with open(temporal, "w", newline="", encoding="utf-8") as archivo:
                escritor = csv.DictWriter(archivo, fieldnames=campos)
                escritor.writeheader()
                escritor.writerows(filas)
            temporal.replace(ruta)
        finally:
            if temporal.exists():
                temporal.unlink()

    def cargar(self) -> None:
        clientes = self._leer("clientes.csv", self.CAMPOS_CLIENTES, Cliente)
        rutas = self._leer("rutas.csv", self.CAMPOS_RUTAS, Ruta)
        pedidos = self._leer("pedidos.csv", self.CAMPOS_PEDIDOS, Pedido)
        for registros, atributo in [(clientes, "id_cliente"), (rutas, "id_ruta"), (pedidos, "folio")]:
            claves = [getattr(registro, atributo) for registro in registros]
            if len(claves) != len(set(claves)):
                raise ValueError("Hay claves duplicadas en los archivos CSV.")
        for pedido in pedidos:
            if not any(c.id_cliente == pedido.id_cliente for c in clientes):
                raise ValueError(f"El cliente de {pedido.folio} no existe.")
            if pedido.id_ruta:
                ruta = next((r for r in rutas if r.id_ruta == pedido.id_ruta), None)
                if ruta is None or (pedido.origen, pedido.destino) != (ruta.origen, ruta.destino):
                    raise ValueError(f"La ruta de {pedido.folio} no existe o no coincide con su trayecto.")
        ultimo = max((int(p.folio.split("-")[1]) for p in pedidos), default=0)
        try:
            with open(self.directorio / "secuencias.csv", "r", newline="", encoding="utf-8") as archivo:
                lector = csv.DictReader(archivo)
                filas = list(lector)
                if lector.fieldnames != ["ultimo_folio"] or len(filas) != 1 or not (filas[0].get("ultimo_folio") or "").isdigit():
                    raise ValueError("El archivo de consecutivos es incorrecto.")
                ultimo = max(ultimo, int(filas[0]["ultimo_folio"]))
        except FileNotFoundError:
            pass
        self.clientes, self.rutas, self.pedidos = clientes, rutas, pedidos
        self.ultimo_folio = ultimo

    def guardar(self) -> None:
        self._escribir("clientes.csv", self.CAMPOS_CLIENTES, [c.to_dict() for c in self.clientes])
        self._escribir("rutas.csv", self.CAMPOS_RUTAS, [r.to_dict() for r in self.rutas])
        self._escribir("secuencias.csv", ["ultimo_folio"], [{"ultimo_folio": self.ultimo_folio}])
        self._escribir("pedidos.csv", self.CAMPOS_PEDIDOS, [p.to_dict() for p in self.pedidos])

    def _guardar_lista(self, entidad, registros) -> None:
        campos = {"clientes": self.CAMPOS_CLIENTES, "rutas": self.CAMPOS_RUTAS, "pedidos": self.CAMPOS_PEDIDOS}
        self._escribir(entidad + ".csv", campos[entidad], [r.to_dict() for r in registros])
        setattr(self, entidad, registros)


gestor = GestorRutasBajio()
print("Clientes cargados al inicio:", len(gestor.clientes))
print("Rutas cargadas al inicio:", len(gestor.rutas))
print("Pedidos cargados al inicio:", len(gestor.pedidos))


Clientes cargados al inicio: 0
Rutas cargadas al inicio: 0
Pedidos cargados al inicio: 0


In [5]:
# CRUD CREATE: crear registros y validar que no se repitan sus identificadores.
# Se amplía la clase del gestor para agregar esta operación.

class GestorRutasBajio(GestorRutasBajio):
    def buscar_cliente(self, id_cliente: str) -> Optional[Cliente]:
        identificador = clave(id_cliente, "CLI", 3)
        for cliente in self.clientes:
            if cliente.id_cliente == identificador:
                return cliente
        return None

    def buscar_ruta(self, id_ruta: str) -> Optional[Ruta]:
        identificador = clave(id_ruta, "RUTA", 2)
        for ruta in self.rutas:
            if ruta.id_ruta == identificador:
                return ruta
        return None

    def buscar_pedido(self, folio: str) -> Optional[Pedido]:
        identificador = clave(folio, "PED", 4)
        for pedido in self.pedidos:
            if pedido.folio == identificador:
                return pedido
        return None

    def crear_cliente(self, cliente: Cliente) -> None:
        if self.buscar_cliente(cliente.id_cliente):
            raise ValueError(f"Ya existe el cliente {cliente.id_cliente}.")
        self._guardar_lista("clientes", self.clientes + [cliente])

    def crear_ruta(self, ruta: Ruta) -> None:
        if self.buscar_ruta(ruta.id_ruta):
            raise ValueError(f"Ya existe la ruta {ruta.id_ruta}.")
        self._guardar_lista("rutas", self.rutas + [ruta])

    def _validar_relaciones(self, pedido: Pedido, disponible: bool = False) -> None:
        if self.buscar_cliente(pedido.id_cliente) is None:
            raise ValueError("El cliente del pedido no existe.")
        if pedido.id_ruta:
            ruta = self.buscar_ruta(pedido.id_ruta)
            if ruta is None:
                raise ValueError("La ruta del pedido no existe.")
            if (pedido.origen, pedido.destino) != (ruta.origen, ruta.destino):
                raise ValueError("La ruta no coincide con el origen y destino del pedido.")
            if disponible and not ruta.disponible:
                raise ValueError("La ruta no está disponible.")

    def crear_pedido(self, pedido: Pedido) -> None:
        if self.buscar_pedido(pedido.folio):
            raise ValueError(f"Ya existe el pedido {pedido.folio}.")
        if pedido.estado != "pendiente":
            raise ValueError("Un pedido nuevo debe estar pendiente.")
        self._validar_relaciones(pedido, disponible=True)
        ultimo = max(self.ultimo_folio, int(pedido.folio.split("-")[1]))
        self._escribir("secuencias.csv", ["ultimo_folio"], [{"ultimo_folio": ultimo}])
        self.ultimo_folio = ultimo
        self._guardar_lista("pedidos", self.pedidos + [pedido])

    def registrar_pedido(self, id_cliente: str, producto: str, cantidad: int, origen: str,
                         destino: str, fecha: str, id_ruta: Optional[str] = None) -> Pedido:
        """Genera el siguiente folio automáticamente para las nuevas solicitudes."""
        pedido = Pedido(f"PED-{self.ultimo_folio + 1:04d}", id_cliente, producto, cantidad,
                        origen, destino, fecha, id_ruta=id_ruta)
        self.crear_pedido(pedido)
        return pedido


# Cargar los CSV; los ejemplos se agregan únicamente si todavía no existen.
gestor = GestorRutasBajio()
clientes_ejemplo = [
    Cliente("CLI-010", "Constructora Guanajuato S.A.", "Guanajuato Capital", "473-100-2233"),
    Cliente("CLI-011", "Ferretera Industrial Guanajuato", "León", "477-200-3344"),
    Cliente("CLI-012", "Aceros del Bajío S.A.", "León", "477-300-5566"),
]
rutas_ejemplo = [
    Ruta("RUTA-01", "León", "Irapuato", 30, 0.6, "Miguel Ángel Torres"),
    Ruta("RUTA-02", "León", "Celaya", 95, 2.0, "Rosa Elena Vázquez"),
    Ruta("RUTA-03", "León", "Guanajuato Capital", 55, 1.5, "Julián Pérez"),
]
pedidos_ejemplo = [
    Pedido("PED-0001", "CLI-010", "Láminas de acero calibre 20", 40,
           "León", "Guanajuato Capital", "2026-09-15", id_ruta="RUTA-03"),
    Pedido("PED-0002", "CLI-011", "Tornillos hexagonales 1/4 pulg", 500,
           "León", "Irapuato", "2026-09-16", id_ruta="RUTA-01"),
    Pedido("PED-0003", "CLI-012", "Cable eléctrico calibre 12 AWG", 200,
           "León", "Celaya", "2026-09-14", id_ruta="RUTA-02"),
    Pedido("PED-0006", "CLI-012", "Alambre recocido", 25,
           "León", "Irapuato", "2026-09-21"),
]
for cliente in clientes_ejemplo:
    if gestor.buscar_cliente(cliente.id_cliente) is None:
        gestor.crear_cliente(cliente)
for ruta in rutas_ejemplo:
    if gestor.buscar_ruta(ruta.id_ruta) is None:
        gestor.crear_ruta(ruta)
for pedido in pedidos_ejemplo:
    if gestor.buscar_pedido(pedido.folio) is None:
        gestor.crear_pedido(pedido)
    else:
        print(f"El pedido {pedido.folio} ya existe; se omite.")
print("Pedidos guardados (se crean pendientes; la salida y entrega se registran en UPDATE):")
for pedido in gestor.pedidos:
    print("-", pedido)


Pedidos guardados (se crean pendientes; la salida y entrega se registran en UPDATE):
- [PED-0001 - CLI-010 - Láminas de acero calibre 20 - 40 - León - Guanajuato Capital - 2026-09-15 - pendiente - RUTA-03]
- [PED-0002 - CLI-011 - Tornillos hexagonales 1/4 pulg - 500 - León - Irapuato - 2026-09-16 - pendiente - RUTA-01]
- [PED-0003 - CLI-012 - Cable eléctrico calibre 12 AWG - 200 - León - Celaya - 2026-09-14 - pendiente - RUTA-02]
- [PED-0006 - CLI-012 - Alambre recocido - 25 - León - Irapuato - 2026-09-21 - pendiente - None]


In [6]:
# CRUD READ: listar registros, filtrar por estado y consultar relaciones.

class GestorRutasBajio(GestorRutasBajio):
    """Extiende el gestor con las operaciones READ del CRUD."""

    def listar_clientes(self) -> list:
        return list(self.clientes)

    def listar_rutas(self) -> list:
        return list(self.rutas)

    def listar_pedidos(self, estado: Optional[str] = None) -> list:
        if estado is not None and estado not in ESTADOS:
            raise ValueError("Estado de consulta inválido.")
        return [p for p in self.pedidos if estado is None or p.estado == estado]

    def pedidos_por_cliente(self, id_cliente: str) -> list:
        cliente = self.buscar_cliente(id_cliente)
        if cliente is None:
            raise ValueError("El cliente no existe.")
        return [p for p in self.pedidos if p.id_cliente == cliente.id_cliente]

    def pendientes_por_transportista(self, transportista: Optional[str] = None) -> dict:
        """Pendientes de entregar: pendientes asignados y en tránsito."""
        resultado = {}
        for ruta in self.rutas:
            if transportista is None or ruta.transportista.casefold() == transportista.strip().casefold():
                resultado.setdefault(ruta.transportista, [])
        for pedido in self.pedidos:
            if pedido.id_ruta and pedido.estado in ("pendiente", "en_transito"):
                nombre = self.buscar_ruta(pedido.id_ruta).transportista
                if nombre in resultado:
                    resultado[nombre].append(pedido)
        return resultado

    def pedidos_sin_ruta(self) -> list:
        return [p for p in self.pedidos if p.estado == "pendiente" and p.id_ruta is None]

    def rutas_compatibles(self, folio: str) -> list:
        pedido = self.buscar_pedido(folio)
        if pedido is None:
            raise ValueError("El pedido no existe.")
        return [r for r in self.rutas if r.disponible and (r.origen, r.destino) == (pedido.origen, pedido.destino)]

    def detalle_pedido(self, folio: str) -> dict:
        pedido = self.buscar_pedido(folio)
        if pedido is None:
            raise ValueError("El pedido no existe.")
        cliente = self.buscar_cliente(pedido.id_cliente)
        ruta = self.buscar_ruta(pedido.id_ruta) if pedido.id_ruta else None
        return {**pedido.to_dict(), "cliente": cliente.nombre, "transportista": ruta.transportista if ruta else None}


# Recrear el gestor para utilizar los métodos agregados en esta celda.
gestor = GestorRutasBajio()
print("READ - Buscar PED-0002:")
pedido_encontrado = gestor.buscar_pedido("PED-0002")
if pedido_encontrado:
    print("-", pedido_encontrado)
else:
    print("Pedido no encontrado.")
print("\nPedidos pendientes:")
for pedido in gestor.listar_pedidos("pendiente"):
    print("-", pedido)
print("\nPedidos de CLI-012:")
for pedido in gestor.pedidos_por_cliente("CLI-012"):
    print("-", pedido)
print("\nPendientes por transportista:")
for nombre, pedidos in gestor.pendientes_por_transportista().items():
    print(nombre, "->", [p.folio for p in pedidos])
print("\nDetalle:", gestor.detalle_pedido("PED-0001"))
print("Sin ruta:", [p.folio for p in gestor.pedidos_sin_ruta()])


READ - Buscar PED-0002:
- [PED-0002 - CLI-011 - Tornillos hexagonales 1/4 pulg - 500 - León - Irapuato - 2026-09-16 - pendiente - RUTA-01]

Pedidos pendientes:
- [PED-0001 - CLI-010 - Láminas de acero calibre 20 - 40 - León - Guanajuato Capital - 2026-09-15 - pendiente - RUTA-03]
- [PED-0002 - CLI-011 - Tornillos hexagonales 1/4 pulg - 500 - León - Irapuato - 2026-09-16 - pendiente - RUTA-01]
- [PED-0003 - CLI-012 - Cable eléctrico calibre 12 AWG - 200 - León - Celaya - 2026-09-14 - pendiente - RUTA-02]
- [PED-0006 - CLI-012 - Alambre recocido - 25 - León - Irapuato - 2026-09-21 - pendiente - None]

Pedidos de CLI-012:
- [PED-0003 - CLI-012 - Cable eléctrico calibre 12 AWG - 200 - León - Celaya - 2026-09-14 - pendiente - RUTA-02]
- [PED-0006 - CLI-012 - Alambre recocido - 25 - León - Irapuato - 2026-09-21 - pendiente - None]

Pendientes por transportista:
Miguel Ángel Torres -> ['PED-0002']
Rosa Elena Vázquez -> ['PED-0003']
Julián Pérez -> ['PED-0001']

Detalle: {'folio': 'PED-0001', 

In [7]:
# CRUD UPDATE: actualizar atributos y controlar los cambios de estado.

class GestorRutasBajio(GestorRutasBajio):
    """Extiende el gestor con la operación UPDATE del CRUD."""

    def actualizar_cliente(self, id_cliente: str, /, **cambios) -> bool:
        cliente = self.buscar_cliente(id_cliente)
        if cliente is None:
            return False
        if set(cambios) - {"nombre", "ciudad", "telefono"}:
            raise ValueError("No se cambia la clave ni se agregan atributos desconocidos.")
        nuevo = Cliente.from_dict({**cliente.to_dict(), **cambios})
        self._guardar_lista("clientes", [nuevo if c.id_cliente == cliente.id_cliente else c for c in self.clientes])
        return True

    def actualizar_ruta(self, id_ruta: str, /, **cambios) -> bool:
        ruta = self.buscar_ruta(id_ruta)
        if ruta is None:
            return False
        if set(cambios) - {"origen", "destino", "distancia_km", "tiempo_horas", "transportista", "disponible"}:
            raise ValueError("No se cambia la clave ni se agregan atributos desconocidos.")
        nueva = Ruta.from_dict({**ruta.to_dict(), **cambios})
        for pedido in self.pedidos:
            if pedido.id_ruta == ruta.id_ruta:
                if (pedido.origen, pedido.destino) != (nueva.origen, nueva.destino):
                    raise ValueError("El trayecto no coincide con los pedidos asociados.")
                if pedido.estado != "pendiente" and any(getattr(ruta, c) != getattr(nueva, c)
                    for c in ("origen", "destino", "distancia_km", "tiempo_horas", "transportista")):
                    raise ValueError("Una ruta con pedidos en tránsito o cerrados solo cambia su disponibilidad.")
        self._guardar_lista("rutas", [nueva if r.id_ruta == ruta.id_ruta else r for r in self.rutas])
        return True

    def actualizar_pedido(self, folio: str, /, **cambios) -> bool:
        """Corrige un pedido pendiente; devuelve False si no se encuentra."""
        pedido = self.buscar_pedido(folio)
        if pedido is None:
            return False
        if pedido.estado != "pendiente":
            raise ValueError("Solo se corrigen datos de pedidos pendientes.")
        if set(cambios) - {"id_cliente", "producto", "cantidad", "origen", "destino", "fecha", "id_ruta"}:
            raise ValueError("No se cambia el folio; usa cambiar_estado para cambiar la situación.")
        nuevo = Pedido.from_dict({**pedido.to_dict(), **cambios})
        self._validar_relaciones(nuevo, disponible=True)
        self._guardar_lista("pedidos", [nuevo if p.folio == pedido.folio else p for p in self.pedidos])
        return True

    def asignar_ruta(self, folio: str, id_ruta: Optional[str]) -> bool:
        return self.actualizar_pedido(folio, id_ruta=id_ruta)

    def cambiar_estado(self, folio: str, estado: str) -> bool:
        pedido = self.buscar_pedido(folio)
        if pedido is None:
            return False
        transiciones = {"pendiente": ("en_transito", "cancelado"), "en_transito": ("entregado",),
                        "entregado": (), "cancelado": ()}
        if estado not in transiciones[pedido.estado]:
            raise ValueError(f"Cambio no permitido: {pedido.estado} -> {estado}.")
        nuevo = Pedido.from_dict({**pedido.to_dict(), "estado": estado})
        self._validar_relaciones(nuevo, disponible=estado == "en_transito")
        self._guardar_lista("pedidos", [nuevo if p.folio == pedido.folio else p for p in self.pedidos])
        return True

    def cancelar_pedido(self, folio: str) -> bool:
        return self.cambiar_estado(folio, "cancelado")


# Prueba de actualización: se carga otra instancia con las nuevas operaciones.
gestor = GestorRutasBajio()
resultado = gestor.actualizar_pedido("PED-0002", cantidad=550)
print("¿Se actualizó PED-0002?", resultado)
print(gestor.buscar_pedido("PED-0002"))
print("Actualizar cliente:", gestor.actualizar_cliente("CLI-011", telefono="477-200-3344"))
print("Disponibilidad de ruta:", gestor.actualizar_ruta("RUTA-01", disponible=True))
if gestor.buscar_pedido("PED-0006").estado == "pendiente":
    gestor.asignar_ruta("PED-0006", "RUTA-01")
print("Ruta asignada:", gestor.buscar_pedido("PED-0006"))

# Registrar las etapas de los ejemplos del caso, sin reabrir pedidos al repetir la celda.
if gestor.buscar_pedido("PED-0001").estado == "pendiente":
    gestor.cambiar_estado("PED-0001", "en_transito")
if gestor.buscar_pedido("PED-0003").estado == "pendiente":
    gestor.cambiar_estado("PED-0003", "en_transito")
if gestor.buscar_pedido("PED-0003").estado == "en_transito":
    gestor.cambiar_estado("PED-0003", "entregado")
print("Salida:", gestor.buscar_pedido("PED-0001"))
print("Entrega confirmada:", gestor.buscar_pedido("PED-0003"))


¿Se actualizó PED-0002? True
[PED-0002 - CLI-011 - Tornillos hexagonales 1/4 pulg - 550 - León - Irapuato - 2026-09-16 - pendiente - RUTA-01]
Actualizar cliente: True
Disponibilidad de ruta: True
Ruta asignada: [PED-0006 - CLI-012 - Alambre recocido - 25 - León - Irapuato - 2026-09-21 - pendiente - RUTA-01]
Salida: [PED-0001 - CLI-010 - Láminas de acero calibre 20 - 40 - León - Guanajuato Capital - 2026-09-15 - en_transito - RUTA-03]
Entrega confirmada: [PED-0003 - CLI-012 - Cable eléctrico calibre 12 AWG - 200 - León - Celaya - 2026-09-14 - entregado - RUTA-02]


In [8]:
# CRUD DELETE: completar la última celda de la plantilla con la baja de registros.

class GestorRutasBajio(GestorRutasBajio):
    """Extiende el gestor con la operación DELETE del CRUD."""

    def eliminar_cliente(self, id_cliente: str) -> bool:
        cliente = self.buscar_cliente(id_cliente)
        if cliente is None:
            return False
        if any(p.id_cliente == cliente.id_cliente for p in self.pedidos):
            raise ValueError("No se elimina un cliente con pedidos asociados.")
        self._guardar_lista("clientes", [c for c in self.clientes if c.id_cliente != cliente.id_cliente])
        return True

    def eliminar_ruta(self, id_ruta: str) -> bool:
        ruta = self.buscar_ruta(id_ruta)
        if ruta is None:
            return False
        if any(p.id_ruta == ruta.id_ruta for p in self.pedidos):
            raise ValueError("No se elimina una ruta con pedidos asociados.")
        self._guardar_lista("rutas", [r for r in self.rutas if r.id_ruta != ruta.id_ruta])
        return True

    def eliminar_pedido(self, folio: str) -> bool:
        pedido = self.buscar_pedido(folio)
        if pedido is None:
            return False
        if pedido.estado not in ("pendiente", "cancelado"):
            raise ValueError("Solo se dan de baja pedidos pendientes o cancelados.")
        self._escribir("secuencias.csv", ["ultimo_folio"], [{"ultimo_folio": self.ultimo_folio}])
        self._guardar_lista("pedidos", [p for p in self.pedidos if p.folio != pedido.folio])
        return True


# Prueba de cancelación y baja usando registros adicionales a los del caso.
gestor = GestorRutasBajio()
if gestor.buscar_cliente("CLI-099") is None:
    gestor.crear_cliente(Cliente("CLI-099", "Cliente de prueba", "Salamanca", "464-100-2233"))
if gestor.buscar_ruta("RUTA-99") is None:
    gestor.crear_ruta(Ruta("RUTA-99", "Salamanca", "Celaya", 50, 1.2, "Transportista de prueba"))
pedido_baja = gestor.registrar_pedido("CLI-099", "Material de prueba", 10,
                                      "Salamanca", "Celaya", "2026-09-22", "RUTA-99")
gestor.cancelar_pedido(pedido_baja.folio)
print("Pedido cancelado:", gestor.buscar_pedido(pedido_baja.folio))
print("¿Se eliminó el pedido?", gestor.eliminar_pedido(pedido_baja.folio))
print("¿Se eliminó la ruta?", gestor.eliminar_ruta("RUTA-99"))
print("¿Se eliminó el cliente?", gestor.eliminar_cliente("CLI-099"))

# Comprobar la persistencia al volver a cargar los archivos.
gestor_recargado = GestorRutasBajio()
assert gestor_recargado.buscar_pedido(pedido_baja.folio) is None
assert gestor_recargado.buscar_pedido("PED-0003").estado == "entregado"
assert gestor_recargado.ultimo_folio == gestor.ultimo_folio
print("\nRegistros recuperados:")
for pedido in gestor_recargado.listar_pedidos():
    print("-", pedido)

# Verificar las situaciones incorrectas descritas en el caso.
def comprobar_error(descripcion, operacion):
    try:
        operacion()
    except ValueError as error:
        print(descripcion, "->", error)
    else:
        raise AssertionError("Se aceptó una operación inválida: " + descripcion)

comprobar_error("Cantidad con letras", lambda: gestor.registrar_pedido(
    "CLI-010", "Varilla", "muchas", "León", "Celaya", "2026-09-22"))
comprobar_error("Ruta incompatible", lambda: gestor.asignar_ruta("PED-0002", "RUTA-03"))
comprobar_error("Cancelar después de salir", lambda: gestor.cancelar_pedido("PED-0001"))
comprobar_error("Reabrir un entregado", lambda: gestor.cambiar_estado("PED-0003", "pendiente"))
comprobar_error("Eliminar cliente con pedidos", lambda: gestor.eliminar_cliente("CLI-010"))
print("\nCRUD completo y persistencia comprobada.")
print("Archivos CSV en la carpeta datos_notebook/.")


Pedido cancelado: [PED-0007 - CLI-099 - Material de prueba - 10 - Salamanca - Celaya - 2026-09-22 - cancelado - RUTA-99]
¿Se eliminó el pedido? True
¿Se eliminó la ruta? True
¿Se eliminó el cliente? True

Registros recuperados:
- [PED-0001 - CLI-010 - Láminas de acero calibre 20 - 40 - León - Guanajuato Capital - 2026-09-15 - en_transito - RUTA-03]
- [PED-0002 - CLI-011 - Tornillos hexagonales 1/4 pulg - 550 - León - Irapuato - 2026-09-16 - pendiente - RUTA-01]
- [PED-0003 - CLI-012 - Cable eléctrico calibre 12 AWG - 200 - León - Celaya - 2026-09-14 - entregado - RUTA-02]
- [PED-0006 - CLI-012 - Alambre recocido - 25 - León - Irapuato - 2026-09-21 - pendiente - RUTA-01]
Cantidad con letras -> La cantidad debe ser un número entero positivo.
Ruta incompatible -> La ruta no coincide con el origen y destino del pedido.
Cancelar después de salir -> Cambio no permitido: en_transito -> cancelado.
Reabrir un entregado -> Cambio no permitido: entregado -> pendiente.
Eliminar cliente con pedidos